# UMAP on Petrophysical Data
## Machine learning for energy applications — Module 4, Exercise 3

**Uniform Manifold Approximation and Projection (UMAP)** is a non-linear
dimensionality-reduction technique that builds a weighted neighbourhood graph of the data
and then optimizes a low-dimensional layout that approximates that graph. It scales well to
large datasets and, unlike scikit-learn's t-SNE, it can transform new rows. Claims that it
preserves global structure better than t-SNE are debated: much of the difference comes from
the initialization.

In this exercise we:
1. Load petrophysical well-log data with facies labels
2. Apply UMAP and visualize the 2D embedding
3. Experiment with `n_neighbors` and `min_dist` hyperparameters
4. Compare UMAP with t-SNE and PCA, scoring every map with trustworthiness at k = 10

> **Note:** UMAP requires the optional `umap-learn` package, listed in
> `requirements-labs-optional.txt` at the course root (`pip install umap-learn`).
> We wrap the import in a try/except block, so the UMAP cells are skipped if it is missing.

> **Exploratory analysis.** The facies labels are kept out of the inputs: they only colour the
> maps after fitting. The scaler and every map are fitted on all 1,000 rows, so the maps describe
> this table. Islands in a map are not evidence that the facies are separable in the logs until
> that is checked in the logs themselves.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE, trustworthiness
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Try importing UMAP — optional package: pip install umap-learn
# (Without ipywidgets, importing umap may print a harmless TqdmWarning about IProgress.)
try:
    import umap
    UMAP_AVAILABLE = True
    print("umap-learn is available.")
except ImportError:
    UMAP_AVAILABLE = False
    print("WARNING: umap-learn is NOT installed.")
    print("Install it with: pip install umap-learn")
    print("UMAP cells will be skipped; PCA and t-SNE will still run.")

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Explore the Data

The dataset contains synthetic petrophysical well-log measurements:
- **log_0 through log_9** — ten well-log channels
- **facies** — lithofacies label (integer class 0, 1 or 2; the file gives no facies names)

In [ ]:
# YOUR CODE HERE
# TODO: Load "petro.csv" into a DataFrame
# TODO: Print the shape and display the first 10 rows


In [ ]:
# YOUR CODE HERE
# TODO: Call df.describe()


In [ ]:
# YOUR CODE HERE
# TODO: Print the facies distribution using df["facies"].value_counts().sort_index()


### 1.1 Separate Features and Labels

The facies label is kept out of the feature matrix `X`: UMAP, t-SNE and PCA are fitted on the
ten logs only, and the labels are used afterwards to colour the maps.

In [ ]:
# YOUR CODE HERE
# TODO: Define feature_cols as all columns except "facies" (the label must not be an input)
# TODO: Extract features X and labels y as numpy arrays
# TODO: Print the number of features, samples, and unique classes

---
## 2 — Standardize Features

The scaler is fitted on all rows because the maps are exploratory. If a UMAP map were reused to
place new wells, the scaler and UMAP would be fitted on training wells only and then applied to
the new wells with `transform`.

In [ ]:
# YOUR CODE HERE
# TODO: Standardize X using StandardScaler
# TODO: Print means and stds after scaling to verify


---
## 3 — Apply UMAP

UMAP has two key hyperparameters:
- **n_neighbors** — controls local vs global structure (similar to perplexity in t-SNE)
- **min_dist** — controls how tightly points are packed in the embedding

We start with default values: `n_neighbors=15`, `min_dist=0.1`. Setting `random_state` makes
the run repeatable but forces single-threaded fitting, so we also pass `n_jobs=1` explicitly.
Every map is scored with trustworthiness at k = 10.

In [ ]:
# YOUR CODE HERE
# TODO: If UMAP_AVAILABLE:
#   - Create umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
#                      random_state=RANDOM_STATE, n_jobs=1)
#   - Fit and transform X_scaled
#   - Print the output shape and trustworthiness(X_scaled, X_umap, n_neighbors=10)
# TODO: Else print a message that UMAP is not installed and set X_umap = None

In [ ]:
# YOUR CODE HERE
# TODO: If UMAP ran successfully, scatter plot the embedding coloured by facies (y)
# Hint: use cmap="tab10" for discrete class colours
# TODO: Add colorbar, axis labels, and title


---
## 4 — Experiment with n_neighbors

- **Small n_neighbors** (e.g., 5): emphasizes very local structure, can fragment clusters
- **Large n_neighbors** (e.g., 50): captures more global structure, smoother embedding

In [ ]:
# YOUR CODE HERE
# TODO: If UMAP_AVAILABLE, loop over nn_values = [5, 15, 50]:
#   - Fit UMAP with each n_neighbors (keep min_dist=0.1, random_state=RANDOM_STATE, n_jobs=1)
#   - Store results in a dictionary umap_nn_results
#   - Print the trustworthiness (n_neighbors=10) of each map

In [ ]:
# YOUR CODE HERE
# TODO: If results exist, create a 1x3 subplot figure
# TODO: For each n_neighbors value, scatter the embedding coloured by facies
# TODO: Add a suptitle


---
## 5 — Experiment with min_dist

- **min_dist = 0.0**: points are packed as tightly as possible, producing very dense clusters
- **min_dist = 0.5**: more spread-out embedding, preserving more of the broad topological structure

In [ ]:
# YOUR CODE HERE
# TODO: If UMAP_AVAILABLE, loop over md_values = [0.0, 0.1, 0.5]:
#   - Fit UMAP with each min_dist (keep n_neighbors=15, random_state=RANDOM_STATE, n_jobs=1)
#   - Store results in a dictionary umap_md_results
#   - Print the trustworthiness (n_neighbors=10) of each map

In [ ]:
# YOUR CODE HERE
# TODO: If results exist, create a 1x3 subplot figure
# TODO: For each min_dist value, scatter the embedding coloured by facies


---
## 6 — Compare UMAP with t-SNE and PCA

Let us apply all three methods and visualize side by side.

In [ ]:
# YOUR CODE HERE
# TODO: Fit PCA with n_components=2 on X_scaled
# TODO: Fit t-SNE with perplexity=30, random_state=RANDOM_STATE, max_iter=1000 on X_scaled
# TODO: Print PCA explained variance and t-SNE KL divergence
# TODO: Compute trustworthiness(X_scaled, embedding, n_neighbors=10) for PCA, t-SNE
#       and (if available) UMAP, store them in a dictionary `trust`, and print them

In [ ]:
# YOUR CODE HERE
# TODO: Create a 1x3 (or 1x2 if UMAP unavailable) subplot figure
#   Panel 1: PCA scatter coloured by facies
#   Panel 2: t-SNE scatter coloured by facies
#   Panel 3: UMAP scatter coloured by facies (if available)
# TODO: Put the trustworthiness of each map in its title
# TODO: Add a suptitle

---
## Summary

In this exercise we applied UMAP to petrophysical data and covered:

1. **Data preparation** — separating features from facies labels and standardizing.
2. **UMAP embedding** — projecting 10 log channels into 2D for visualization.
3. **n_neighbors tuning** — small values emphasize local structure; large values let broader structure shape the layout.
4. **min_dist tuning** — controls cluster tightness in the embedding.
5. **Method comparison** — PCA (linear, fast, interpretable), t-SNE (non-linear, local focus, no transform for new rows), UMAP (non-linear, neighbourhood graph, can transform new rows), each scored with trustworthiness at k = 10.

**Key takeaway**: UMAP is a useful tool for visualizing high-dimensional petrophysical data,
and it can place new rows on an existing map. Like t-SNE, its axes are not interpretable and
its pictures depend on `n_neighbors`, `min_dist` and the seed. Well-separated islands in a map
coloured by facies are not proof that the facies are separable in the logs: report the settings
and the trustworthiness, repeat settings and seeds, and confirm any group in the original logs.